In [ ]:
# Step 1: Mount your Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import tensorflow

In [ ]:
# import os
# from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img, array_to_img

# # Set paths
# input_folder = '/content/drive/MyDrive/dxtaset/real_bills/'
# output_folder = '/content/drive/MyDrive/dxtaset/augmented_real_bills/'

# # Make sure output folder exists
# os.makedirs(output_folder, exist_ok=True)

# # Set up the ImageDataGenerator for augmentation
# datagen = ImageDataGenerator(
#     rotation_range=40,
#     width_shift_range=0.2,
#     height_shift_range=0.2,
#     shear_range=0.2,
#     zoom_range=0.2,
#     horizontal_flip=True,
#     fill_mode='nearest'
# )

# # Loop through each image in the input folder
# for filename in os.listdir(input_folder):
#     if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
#         img_path = os.path.join(input_folder, filename)
#         img = load_img(img_path)  # Load image as PIL
#         x = img_to_array(img)     # Convert to Numpy array
#         x = x.reshape((1,) + x.shape)  # Reshape to (1, height, width, channels)

#         # Generate and save 10 augmented images
#         i = 0
#         for batch in datagen.flow(x, batch_size=1, save_to_dir=output_folder,
#                                   save_prefix='aug', save_format='jpg'):
#             i += 1
#             if i >= 10:  # generate 10 augmented images per original
#                 break

# print("✅ Augmentation complete. Check your 'augmented_real_bills' folder in Drive.")


✅ Augmentation complete. Check your 'augmented_real_bills' folder in Drive.


In [ ]:
# import os
# from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img, array_to_img

# # Set paths
# input_folder = '/content/drive/MyDrive/dxtaset/100counterfeit/'
# output_folder = '/content/drive/MyDrive/dxtaset/augmented_counterfeit_bills/'

# # Make sure output folder exists
# os.makedirs(output_folder, exist_ok=True)

# # Set up the ImageDataGenerator for augmentation
# datagen = ImageDataGenerator(
#     rotation_range=40,
#     width_shift_range=0.2,
#     height_shift_range=0.2,
#     shear_range=0.2,
#     zoom_range=0.2,
#     horizontal_flip=True,
#     fill_mode='nearest'
# )

# # Loop through each image in the input folder
# for filename in os.listdir(input_folder):
#     if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
#         img_path = os.path.join(input_folder, filename)
#         img = load_img(img_path)  # Load image as PIL
#         x = img_to_array(img)     # Convert to Numpy array
#         x = x.reshape((1,) + x.shape)  # Reshape to (1, height, width, channels)

#         # Generate and save 10 augmented images
#         i = 0
#         for batch in datagen.flow(x, batch_size=1, save_to_dir=output_folder,
#                                   save_prefix='aug', save_format='jpg'):
#             i += 1
#             if i >= 10:  # generate 10 augmented images per original
#                 break

# print("✅ Augmentation complete. Check your 'augmented_real_bills' folder in Drive.")


✅ Augmentation complete. Check your 'augmented_real_bills' folder in Drive.


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D
from keras.layers import Activation, Dropout, Flatten, Dense

model = Sequential()
model.add(Conv2D(32, (3, 3), input_shape=(150,150,3)))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Conv2D(32, (3, 3)))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Conv2D(64, (3, 3)))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))



/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.add(Flatten())
model.add(Dense(64))
model.add(Activation('relu'))
model.add(Dropout(0.5))
model.add(Dense(1))
model.add(Activation('sigmoid'))

model.compile(loss='binary_crossentropy',
              optimizer='rmsprop',
              metrics=['accuracy'])

In [ ]:
batch_size = 16

# this is the augmentation configuration we will use for training
train_datagen = ImageDataGenerator(
        rescale=1./255,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True)

# this is the augmentation configuration we will use for testing:
# only rescaling
test_datagen = ImageDataGenerator(rescale=1./255)

# this is a generator that will read pictures found in
# subfolers of 'data/train', and indefinitely generate
# batches of augmented image data
train_generator = train_datagen.flow_from_directory(
        '/content/drive/MyDrive/cxrrencydxtaset/training',  # this is the target directory
        target_size=(150, 150),  # all images will be resized to 150x150
        batch_size=batch_size,
        class_mode='binary')  # since we use binary_crossentropy loss, we need binary labels

# this is a similar generator, for validation data
validation_generator = test_datagen.flow_from_directory(
        '/content/drive/MyDrive/cxrrencydxtaset/testing',
        target_size=(150, 150),
        batch_size=batch_size,
        class_mode='binary')

Found 2188 images belonging to 2 classes.
Found 549 images belonging to 2 classes.


In [ ]:
model.fit(
    train_generator,
    steps_per_epoch=2000 // batch_size,
    epochs=50,
    validation_data=validation_generator,
    validation_steps=800 // batch_size
)
model.save_weights('first_try.weights.h5')



Epoch 1/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 242ms/step - accuracy: 0.8446 - loss: 0.3840

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/epoch_iterator.py:107: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


125/125 ━━━━━━━━━━━━━━━━━━━━ 172s 1s/step - accuracy: 0.8447 - loss: 0.3839 - val_accuracy: 0.8306 - val_loss: 0.3738
Epoch 2/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 37ms/step - accuracy: 0.9116 - loss: 0.2111 - val_accuracy: 0.6630 - val_loss: 1.0906
Epoch 3/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 23s 182ms/step - accuracy: 0.8651 - loss: 0.3153 - val_accuracy: 0.8324 - val_loss: 0.3709
Epoch 4/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 39ms/step - accuracy: 0.8933 - loss: 0.2255 - val_accuracy: 0.5719 - val_loss: 1.3548
Epoch 5/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 38s 196ms/step - accuracy: 0.8944 - loss: 0.2645 - val_accuracy: 0.8106 - val_loss: 0.5486
Epoch 6/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 32ms/step - accuracy: 0.8942 - loss: 0.2471 - val_accuracy: 0.7523 - val_loss: 0.5994
Epoch 7/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 35s 182ms/step - accuracy: 0.9163 - loss: 0.2151 - val_accuracy: 0.9089 - val_loss: 0.2387
Epoch 8/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - accuracy: 0.9143 - loss: 0.2260 - val_accuracy:

In [ ]:
model.save('/content/drive/MyDrive/custom_cnn_model.h5')

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 148, 148, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 72, 72, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 72, 72, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 34, 34, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 34, 34, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 17, 17, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 18496)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     1,183,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 1)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,425,028 (9.25 MB)

 Trainable params: 1,212,513 (4.63 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,212,515 (4.63 MB)

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image


test_image = image.load_img('/content/drive/MyDrive/IMG_5476.jpg', target_size=(150, 150))
test_image = image.img_to_array(test_image)
test_image = test_image / 255.0
test_image = np.expand_dims(test_image, axis=0)


result = model.predict(test_image)


if result[0][0] >= 0.5:
    prediction = 'real'
else:
    prediction = 'counterfeit'

print(prediction)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 649ms/step
real


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image
czz = load_model("/content/drive/MyDrive/custom_cnn_model.h5")


test_image = image.load_img('/content/drive/MyDrive/cxnterfit.jpg', target_size=(150, 150))
test_image = image.img_to_array(test_image)
test_image = test_image / 255.0
test_image = np.expand_dims(test_image, axis=0)


result = model.predict(test_image)

if result[0][0] >= 0.5:
    prediction = 'real'
else:
    prediction = 'counterfeit'

print(prediction)

NameError: name 'load_model' is not defined

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image


test_image = image.load_img('/content/drive/MyDrive/counterfxt.jpg', target_size=(150, 150))
test_image = image.img_to_array(test_image)
test_image = test_image / 255.0
test_image = np.expand_dims(test_image, axis=0)

result = model.predict(test_image)


if result[0][0] >= 0.5:
    prediction = 'real'
else:
    prediction = 'counterfeit'

print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
counterfeit


In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model

In [ ]:
resnet_base = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(150, 150, 3)
)
resnet_base.trainable = False

resnet_base.trainable = False

x = resnet_base.output
x = GlobalAveragePooling2D()(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)

resnet_model = Model(inputs=resnet_base.input, outputs=output)

resnet_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
history_resnet = resnet_model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // batch_size,
    epochs=10,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // batch_size
)

Epoch 1/10
136/136 ━━━━━━━━━━━━━━━━━━━━ 46s 248ms/step - accuracy: 0.5559 - loss: 0.7226 - val_accuracy: 0.5901 - val_loss: 0.6359
Epoch 2/10
136/136 ━━━━━━━━━━━━━━━━━━━━ 4s 27ms/step - accuracy: 0.8125 - loss: 0.5341 - val_accuracy: 0.5699 - val_loss: 0.6490
Epoch 3/10
136/136 ━━━━━━━━━━━━━━━━━━━━ 67s 199ms/step - accuracy: 0.6231 - loss: 0.6253 - val_accuracy: 0.6636 - val_loss: 0.6004
Epoch 4/10
136/136 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.5625 - loss: 0.6161 - val_accuracy: 0.6654 - val_loss: 0.6007
Epoch 5/10
136/136 ━━━━━━━━━━━━━━━━━━━━ 37s 192ms/step - accuracy: 0.6516 - loss: 0.6181 - val_accuracy: 0.6452 - val_loss: 0.6145
Epoch 6/10
136/136 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.5000 - loss: 0.6728 - val_accuracy: 0.6342 - val_loss: 0.6207
Epoch 7/10
136/136 ━━━━━━━━━━━━━━━━━━━━ 41s 302ms/step - accuracy: 0.6496 - loss: 0.6016 - val_accuracy: 0.6471 - val_loss: 0.6130
Epoch 8/10
136/136 ━━━━━━━━━━━━━━━━━━━━ 4s 26ms/step - accuracy: 0.6875 - loss: 0.5883 - 

In [ ]:
resnet_model.save('/content/drive/MyDrive/resnet50_model.h5')


In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np

# Load the trained models
cnn_model = load_model("/content/drive/MyDrive/custom_cnn_model.h5")
resnet_model = load_model("/content/drive/MyDrive/resnet50_model.h5")

def preprocess_img(img_path):
    img = image.load_img(img_path, target_size=(150, 150))
    img = image.img_to_array(img)
    img = img / 255.0  # normalize
    img = np.expand_dims(img, axis=0)
    return img

def ensemble_predict(img_path, threshold=0.5):
    img = preprocess_img(img_path)
    pred_cnn = cnn_model.predict(img)[0][0]
    pred_resnet = resnet_model.predict(img)[0][0]
    avg_pred = (pred_cnn + pred_resnet) / 2
    label = 'real' if avg_pred >= threshold else 'counterfeit'
    print(f"Custom CNN: {pred_cnn:.4f}, ResNet50: {pred_resnet:.4f}, Avg: {avg_pred:.4f}")
    print(f"Final Ensemble Prediction: {label}")
    return label

In [ ]:
ensemble_predict('/content/drive/MyDrive/cxnterfit.jpg')

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
Custom CNN: 0.0000, ResNet50: 0.4151, Avg: 0.2076
Final Ensemble Prediction: counterfeit


'counterfeit'

In [ ]:
ensemble_predict('/content/drive/MyDrive/IMG_5476.jpg')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Custom CNN: 1.0000, ResNet50: 0.4493, Avg: 0.7247
Final Ensemble Prediction: real


'real'